# 04_texto_transformers — Fine-Tuning con Transformers
Este notebook realiza fine-tuning de un modelo Transformer preentrenado para clasificación de emociones en texto y guarda el modelo ajustado.

**Modelo base:** Transformer preentrenado (HuggingFace)  
**Salida:** `../models/`

In [1]:
!pip install -U pip

# Torch >= 2.6 + CUDA
!pip install -U torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu124

# Transformers y amigos (muy importante que esto quede actualizado)
!pip install -U transformers datasets accelerate evaluate huggingface_hub safetensors ipywidgets jupyterlab_widgets

print("Instalación completa. Ahora reinicia kernel.")

Looking in indexes: https://download.pytorch.org/whl/cu124
Instalación completa. Ahora reinicia kernel.


In [2]:
print("Ahora reinicia el kernel: Kernel → Restart Kernel (IMPORTANTE)")


Ahora reinicia el kernel: Kernel → Restart Kernel (IMPORTANTE)


In [3]:
import torch
import transformers

print("torch:", torch.__version__)
print("transformers:", transformers.__version__)
print("CUDA disponible:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

torch: 2.6.0+cu124
transformers: 5.1.0
CUDA disponible: True
GPU: NVIDIA GeForce RTX 3060


## 1) Cargar dataset limpio

Se carga el dataset previamente limpiado y preparado para entrenamiento con Transformers.  
Este dataset contiene el texto y su etiqueta de emoción correspondiente.

In [4]:
import pandas as pd
from sklearn.model_selection import train_test_split

CSV_PATH = "../data/text/tweets_sentiment_clean.csv"

df = pd.read_csv(CSV_PATH, encoding="utf-8")

df = df.dropna(subset=["texto", "label"]).copy()
df["texto"] = df["texto"].astype(str).str.strip()
df = df[df["texto"] != ""]
df = df[df["texto"].str.len() >= 3]

print("Filas:", len(df))
print(df["label"].value_counts())
df.head(3)

Filas: 132681
label
NEU    112188
POS     11004
NEG      9489
Name: count, dtype: int64


,texto,label
0,"Alisson puede estar más tranquilo, no cargará ...",POS
1,Es que el director ejecutivo es mujer. So... J...,NEU
2,Upto £100 freebets lol gunners matchedbetting ...,NEU


## 2) División train/val (estratificada)

Se divide el dataset en conjunto de entrenamiento y validación manteniendo la proporción original de clases mediante `stratify`.  
Esto permite evaluar el modelo de forma equilibrada y fiable.

In [5]:
train_df, val_df = train_test_split(
    df,
    test_size=0.1,
    random_state=42,
    stratify=df["label"]
)

print("Train:", len(train_df), "Val:", len(val_df))

Train: 119412 Val: 13269


## 2) Preparar etiquetas

Las etiquetas de emoción se convierten a formato numérico para que el modelo pueda trabajar con ellas.  
Se define el mapeo `label ↔ id` necesario para el entrenamiento.

In [6]:
from datasets import Dataset

label2id = {"NEG": 0, "NEU": 1, "POS": 2}
id2label = {v: k for k, v in label2id.items()}

def add_labels(ex):
    ex["labels"] = label2id[ex["label"]]
    return ex

train_ds = Dataset.from_pandas(train_df.reset_index(drop=True)).map(add_labels)
val_ds   = Dataset.from_pandas(val_df.reset_index(drop=True)).map(add_labels)

train_ds

Map:   0%|          | 0/119412 [00:00<?, ? examples/s]

Map:   0%|          | 0/13269 [00:00<?, ? examples/s]

Dataset({
    features: ['texto', 'label', 'labels'],
    num_rows: 119412
})

## 4) Tokenización con el modelo preentrenado

Se utiliza el tokenizer oficial del modelo Transformer para convertir el texto en representaciones numéricas.  
Se aplican padding y truncado para asegurar una longitud uniforme en los lotes de entrenamiento.

In [7]:
from transformers import AutoTokenizer

MODEL_NAME = "dccuchile/bert-base-spanish-wwm-cased"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tok(batch):
    texts = [str(x) if x is not None else "" for x in batch["texto"]]
    return tokenizer(texts, truncation=True, padding="max_length", max_length=128)

train_ds = train_ds.map(tok, batched=True)
val_ds   = val_ds.map(tok, batched=True)

train_ds.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])
val_ds.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])

train_ds[0]

Map:   0%|          | 0/119412 [00:00<?, ? examples/s]

Map:   0%|          | 0/13269 [00:00<?, ? examples/s]

{'labels': tensor(1),
 'input_ids': tensor([    4,  1773,  1266,  2367, 12829,  1674, 28171,  1110,  4516,  1359,
          2551, 27745,  2081,  1046,  3393,  1009,  6567,  8120,  1063,  1025,
          1095,  2218,  1040,  5742,  1072,  2520,  1009, 12494, 30961, 11680,
         30970,     5,     1,     1,     1,     1,     1,     1,     1,     1,
             1,     1,     1,     1,     1,     1,     1,     1,     1,     1,
             1,     1,     1,     1,     1,     1,     1,     1,     1,     1,
             1,     1,     1,     1,     1,     1,     1,     1,     1,     1,
             1,     1,     1,     1,     1,     1,     1,     1,     1,     1,
             1,     1,     1,     1,     1,     1,     1,     1,     1,     1,
             1,     1,     1,     1,     1,     1,     1,     1,     1,     1,
             1,     1,     1,     1,     1,     1,     1,     1,     1,     1,
             1,     1,     1,     1,     1,     1,     1,     1,     1,     1,
             1,  

## 5) Definir modelo y parámetros de entrenamiento

Se carga el modelo preentrenado adaptado para clasificación de secuencias.  
Se configuran los principales hiperparámetros de entrenamiento:

- Learning rate  
- Batch size  
- Número de épocas  
- Weight decay

In [9]:
import numpy as np
import evaluate
import transformers
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer

accuracy = evaluate.load("accuracy")
f1 = evaluate.load("f1")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy.compute(predictions=preds, references=labels)["accuracy"],
        "f1_macro": f1.compute(predictions=preds, references=labels, average="macro")["f1"],
    }

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=3,
    label2id=label2id,
    id2label=id2label,
    use_safetensors=True
)

# ---- TrainingArguments compatible ----
common_args = dict(
    output_dir="../models/text_beto_v1",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=3,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    fp16=True,
    logging_steps=50,
    report_to="none",
)

# Detecta qué nombre usa tu versión: evaluation_strategy o eval_strategy
sig = TrainingArguments.__init__.__code__.co_varnames
if "evaluation_strategy" in sig:
    common_args["evaluation_strategy"] = "epoch"
    common_args["save_strategy"] = "epoch"
elif "eval_strategy" in sig:
    common_args["eval_strategy"] = "epoch"
    common_args["save_strategy"] = "epoch"
else:
    # Si tu versión es muy vieja, quitamos evaluación automática
    print("⚠️ Tu transformers es muy antiguo: sin eval/saving por epoch. Se entrenará igual.")
    
args = TrainingArguments(**common_args)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    compute_metrics=compute_metrics
)

trainer.train()

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: dccuchile/bert-base-spanish-wwm-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
bert.embeddings.position_ids               | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
bert.pooler.dense.weight                   | MISSING    | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 
bert.pooler.dense.bias                     | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. C

Epoch,Training Loss,Validation Loss,Accuracy,F1 Macro
1,0.246304,0.246550,0.906097,0.749606
2,0.161563,0.288645,0.906248,0.752871
3,0.109298,0.380925,0.906624,0.763293


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La

TrainOutput(global_step=22392, training_loss=0.2013483678455564, metrics={'train_runtime': 2836.9587, 'train_samples_per_second': 126.275, 'train_steps_per_second': 7.893, 'total_flos': 2.356417457830195e+16, 'train_loss': 0.2013483678455564, 'epoch': 3.0})

## 6) Guardar modelo entrenado

El modelo ajustado y su tokenizer se guardan en la carpeta de modelos para poder reutilizarlos posteriormente en inferencia o en la interfaz gráfica.

In [10]:
OUT_DIR = "../models/text_beto_v1/best"
trainer.save_model(OUT_DIR)
tokenizer.save_pretrained(OUT_DIR)
print("Guardado en:", OUT_DIR)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Guardado en: ../models/text_beto_v1/best


## 7) Evaluación del modelo

Se evalúa el rendimiento del modelo sobre el conjunto de validación utilizando métricas como accuracy y F1-score.  
Esto permite medir la capacidad de generalización del modelo.

In [11]:
metrics = trainer.evaluate()
metrics

{'eval_loss': 0.38092541694641113,
 'eval_accuracy': 0.9066244630341397,
 'eval_f1_macro': 0.7632931961334873,
 'eval_runtime': 19.8136,
 'eval_samples_per_second': 669.69,
 'eval_steps_per_second': 20.945,
 'epoch': 3.0}

In [1]:
import pandas as pd
df = pd.read_csv("../data/audio/manifest_es.csv")
print(df.columns)
df.head()

Index(['path', 'label', 'dataset', 'speaker', 'split', 'utt_id', 'level',
       'lang'],
      dtype='str')


,path,label,dataset,speaker,split,utt_id,level,lang
0,C:\Users\Rafa\Downloads\proyecto-ia-20260212T1...,anger,mesd_es,mesd_C_A,all,Anger_C_A_abajo,NaN,NaN
1,C:\Users\Rafa\Downloads\proyecto-ia-20260212T1...,anger,mesd_es,mesd_C_A,all,Anger_C_A_adios,NaN,NaN
2,C:\Users\Rafa\Downloads\proyecto-ia-20260212T1...,anger,mesd_es,mesd_C_A,all,Anger_C_A_antes,NaN,NaN
3,C:\Users\Rafa\Downloads\proyecto-ia-20260212T1...,anger,mesd_es,mesd_C_A,all,Anger_C_A_arriba,NaN,NaN
4,C:\Users\Rafa\Downloads\proyecto-ia-20260212T1...,anger,mesd_es,mesd_C_A,all,Anger_C_A_ayer,NaN,NaN
